# Technical Explainer — Tesla production package

Use **Runtime → Change runtime type → GPU** if available. Upload the matching ZIP below. This notebook reuses frozen narration and completed clips; it needs no AI API key. CPU fallback is supported. A successful encode still needs actual final visual review.

The package snapshot may have more completed chapters than the original preview. Colab execution itself has not been verified from this chat; the adapter has a local CPU render smoke test.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one project ZIP.')
snapshot_id = hashlib.sha256(Path(archives[0]).read_bytes()).hexdigest()[:12]
PACKAGE_ROOT = Path('/content') / ('tesla_explainer_' + snapshot_id)
PACKAGE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as bundle:
    entries = bundle.infolist()
    if len(entries) > 2500 or sum(x.file_size for x in entries) > 2 * 1024**3:
        raise ValueError('Archive exceeds the expected package limits.')
    for item in entries:
        target = (PACKAGE_ROOT / item.filename).resolve()
        if not target.is_relative_to(PACKAGE_ROOT.resolve()) or ((item.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('Unsafe archive entry: ' + item.filename)
    bundle.extractall(PACKAGE_ROOT)
print('Extracted:', PACKAGE_ROOT)


## Install the matching runtime

This creates a separate Python 3.13 environment. Frozen narration is already included, so the voice weights are unnecessary for this resume operation.

In [ ]:
subprocess.run([sys.executable, str(PACKAGE_ROOT / 'explainer/colab/bootstrap.py')], cwd=PACKAGE_ROOT, check=True)
RENDER_PYTHON = PACKAGE_ROOT / '.colab-render-venv/bin/python'


## Verify the render device

This renders a tiny diagnostic, rather than assuming a listed GPU works. If the GPU smoke test fails, set `USE_GPU = False` and rerun this cell for CPU rendering.

In [ ]:
USE_GPU = True
render_env = dict(os.environ, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
render_env['EXPLAINER_FORCE_CPU'] = '0' if USE_GPU else '1'
subprocess.run([str(RENDER_PYTHON), '-m', 'explainer.colab.runtime', 'smoke'], cwd=PACKAGE_ROOT, env=render_env, check=True)


## See what will be reused

In [ ]:
project = PACKAGE_ROOT / 'explainer/projects/tesla-hw3'
production = project / 'build/full-v1'
manifest = json.loads((production / 'manifest.json').read_text())
for chapter in manifest['shots']:
    if chapter['index'] == 0:
        status = 'approved opening — reuse'
    else:
        folder = production / chapter['id']
        qa = folder / 'qa.json'
        ready = qa.exists() and (folder / 'chapter.mp4').exists() and json.loads(qa.read_text()).get('passed')
        status = 'completed clip — reuse' if ready else 'render missing chapter'
    print(f"{chapter['index'] + 1:02d}  {chapter['title']}: {status}")


## Resume production

The source and media checksums are validated first. Missing chapters are rendered with the detected device, encoded with the frozen narration, then assembled and checked. No footage is published automatically.

In [ ]:
subprocess.run([str(RENDER_PYTHON), '-m', 'explainer.colab.runtime', 'resume'], cwd=PACKAGE_ROOT, env=render_env, check=True)


## Watch and inspect the assembled film

In [ ]:
from IPython.display import Video, display
video = production / 'tesla-planning-complete.mp4'
display(Video(str(video), embed=True, width=960))
print('Inspection sheets:', production / 'final-inspection')
print('Master JSON:', production / 'tesla-project.json')


## Download outputs

Run either download line when ready. The JSON describes the project and includes procedural source; audio/video remain separate files.

In [ ]:
files.download(str(production / 'tesla-planning-complete.mp4'))


In [ ]:
files.download(str(production / 'tesla-project.json'))
